# Piedmontese Wikipedia corpus

Builds the monolingual Piedmontese corpus used for denoising pretraining (DAE) of NLLB, starting from the Wikipedia dump of 1 February 2026.

`dump.xml.bz2` → `pmswiki_filtered.jsonl` → `pmswiki_clean.jsonl` → `pmswiki_noleak.jsonl`

The final `pmswiki_noleak.jsonl` is published on Hugging Face ([offelia39/pms-wiki-mined-corpora](https://huggingface.co/datasets/offelia39/pms-wiki-mined-corpora)); the fine-tuning notebook reads it from there.
Code for the rest of the project: [panchenko-ling/piedmontese_wiki_ml](https://github.com/panchenko-ling/piedmontese_wiki_ml).

Always run **§0**. Every later section reads its input from Drive, so it can be run on its own.

## §0 Setup

Paths assume everything lives in one Drive folder; change `BASE` if yours is different.
The dump (`pmswiki-20260201-pages-articles.xml.bz2`) comes from [dumps.wikimedia.org/pmswiki](https://dumps.wikimedia.org/pmswiki/).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q mwxml mwparserfromhell

In [ ]:
import bz2
import csv
import json
import re

import mwxml             # Wikimedia XML dump parser
import mwparserfromhell  # wikitext parser
from tqdm import tqdm

In [ ]:
BASE = "/content/drive/MyDrive/Tesi_piedmontese/"

DUMP     = BASE + "pmswiki-20260201-pages-articles.xml.bz2"
FILTERED = BASE + "pmswiki_filtered.jsonl"
CLEAN    = BASE + "pmswiki_clean.jsonl"
NOLEAK   = BASE + "pmswiki_noleak.jsonl"
MINED    = BASE + "wiki_mined/wiki_mined.csv"   # output of the Wiki miner notebook

MIN_WORDS = 30   # shorter documents are dropped

## §1 Dump filtering
in: `pmswiki-20260201-pages-articles.xml.bz2` · out: `pmswiki_filtered.jsonl`

*Can be skipped if `pmswiki_filtered.jsonl` is saved on Drive.*

Only main-namespace articles are kept. Pages are dropped if they are tagged as AI-written (`Categorìa:Artìcoj scrivù con l'AI`), are disambiguation pages or redirects, or are marked as stubs. Wikitext is kept raw at this stage.

In [ ]:
def exclusion_reason(wikitext):
    """Return why a page should be dropped, or None to keep it."""
    low = wikitext.lower()
    if "Categorìa:Artìcoj scrivù con l'AI" in wikitext:
        return "ai"
    if "disambigua" in low:
        return "disambiguation"
    if wikitext.strip().startswith("#REDIRECT"):
        return "redirect"
    if "{{stub}}" in low:
        return "stub"
    return None

In [ ]:
def filter_dump(input_path, output_path):
    stats = {"total_pages": 0, "kept": 0, "empty": 0,
             "ai": 0, "disambiguation": 0, "redirect": 0, "stub": 0}

    with bz2.open(input_path, "rb") as dump_file, \
         open(output_path, "w", encoding="utf-8") as out:
        dump = mwxml.Dump.from_file(dump_file)

        for page in tqdm(dump.pages, desc="Pages"):
            stats["total_pages"] += 1

            if page.namespace != 0:   # main namespace only
                continue
            if page.id == 1:          # main page
                continue

            revision = next(iter(page), None)   # pages-articles dumps hold one revision
            raw_text = revision.text if revision else None

            if raw_text is None:
                stats["empty"] += 1
                continue

            reason = exclusion_reason(raw_text)
            if reason:
                stats[reason] += 1
                continue

            out.write(json.dumps({"id": page.id, "title": page.title,
                                  "raw_text": raw_text}, ensure_ascii=False) + "\n")
            stats["kept"] += 1

    return stats

In [ ]:
stats = filter_dump(DUMP, FILTERED)
stats

## §2 Markup cleaning
in: `pmswiki_filtered.jsonl` · out: `pmswiki_clean.jsonl`

*Can be skipped if `pmswiki_clean.jsonl` is saved on Drive.*

`mwparserfromhell` strips most of the markup. A series of regexes then removes what it leaves behind: category and file links, URLs, reference section titles, image caption options (`thumb`, `destra`, `200px`…), stray pipes, ISO code boilerplate and `[= …]` blocks. Documents shorter than 30 words after cleaning are dropped.

In [ ]:
def clean_wikitext(text):
    """Enhanced cleaning targeting remaining wiki markup issues."""

    wikicode = mwparserfromhell.parse(text)

    # Remove categories explicitly
    for category in wikicode.filter_wikilinks():
        if str(category.title).startswith("Categorìa:"):
            try:
                wikicode.remove(category)
            except ValueError:
                pass

    # Remove file/image links
    nodes_to_remove = []
    for link in wikicode.filter_wikilinks():
        title = str(link.title).lower()
        if (
            title.startswith("file:")
            or title.startswith("image:")
            or title.startswith("immagine:")
            or title.startswith("fitxer:")
        ):
            nodes_to_remove.append(link)

    for node in nodes_to_remove:
        try:
            wikicode.remove(node)
        except ValueError:
            pass

    # Strip wiki markup
    cleaned = wikicode.strip_code(normalize=True, collapse=True)

    # Remove URLs - MUST be before pipe cleaning (URLs contain pipes/brackets)
    cleaned = re.sub(r'https?://\S+', '', cleaned)
    cleaned = re.sub(r'www\.\S+', '', cleaned)

    # Remove common wiki section headers EARLY (before pipe cleanup)
    cleaned = re.sub(
        r'\b(?:Anliure esterne|Sit istitussional|Colegament estern|Arferiment|Vod autis|'
        r'Referenze|Nòte|Bibliografìa|Collegamenti esterni|External links|Références|See also|'
        r'Ligação externa|Ligações externas)\b',
        '',
        cleaned,
        flags=re.IGNORECASE
    )

    # Remove image caption directives with all spacing patterns
    cleaned = re.sub(
        r'\|\s*(?:thumb|thumbnail|frame|frameless|left|right|sinistra|destra|center|centre|miniatura)\s*\|',
        ' ',
        cleaned,
        flags=re.IGNORECASE
    )

    # Remove caption keywords followed by pipe (handles "thumbnail|destra|", "right|thumb", etc.)
    cleaned = re.sub(
        r'\b(?:thumb|thumbnail|frame|frameless|left|right|sinistra|destra|center|centre|miniatura)\s*\|+',
        '',
        cleaned,
        flags=re.IGNORECASE
    )

    # Remove standalone caption keywords
    cleaned = re.sub(r'\bthumb\b', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\bthumbnail\b', '', cleaned, flags=re.IGNORECASE)

    # Remove pixel dimensions
    cleaned = re.sub(r'\b\d+\s*px\b', '', cleaned, flags=re.IGNORECASE)

    # Remove leading pipes on line start
    cleaned = re.sub(r'^\s*\|\|?\s+', '', cleaned, flags=re.MULTILINE)

    # Remove ALL orphaned/double pipes (|| or | before non-space)
    cleaned = re.sub(r'\|\|+', '', cleaned)
    cleaned = re.sub(r'\|\s*([A-Za-zàèéìòùäëïöüâêîôûãõ])', r'\1', cleaned)

    # Remove isolated pipes with surrounding whitespace
    cleaned = re.sub(r'\s+\|\s+', ' ', cleaned)

    # Remove remaining single pipes (edge case: | followed by space or end)
    cleaned = re.sub(r'\|\s+', '', cleaned)
    cleaned = re.sub(r'\s+\|', '', cleaned)

    # Remove ISO language codes and bibliography references
    cleaned = re.sub(
        r'(?:C\Sdes ISO|ISO 639|Arferiment bibliogràfich|https?://www\.sil\.org).*',
        '',
        cleaned,
        flags=re.IGNORECASE
    )

    # Remove [= ...] synonym/description blocks
    cleaned = re.sub(r'\[=.*?\]', '', cleaned, flags=re.DOTALL)

    # Normalize whitespace
    cleaned = " ".join(cleaned.split())

    return cleaned

In [ ]:
kept, removed_short = 0, 0

with open(FILTERED, encoding="utf-8") as infile, \
     open(CLEAN, "w", encoding="utf-8") as outfile:
    for line in tqdm(infile):
        article = json.loads(line)
        text = clean_wikitext(article["raw_text"])

        if len(text.split()) < MIN_WORDS:
            removed_short += 1
            continue

        outfile.write(json.dumps({"id": article["id"], "title": article["title"],
                                  "text": text}, ensure_ascii=False) + "\n")
        kept += 1

print("Kept:", kept)
print("Removed (too short):", removed_short)

## §3 Removing mined sentences
in: `pmswiki_clean.jsonl`, `wiki_mined/wiki_mined.csv` · out: `pmswiki_noleak.jsonl`

*Can be skipped if `pmswiki_noleak.jsonl` is saved on Drive.*

The parallel corpus was mined from the same Wikipedia, so every Piedmontese sentence that ended up in it is removed here; otherwise DAE pretraining would already have seen the MT targets. The unfiltered mined file is used on purpose: `wiki_mined_clean.csv` is a subset of it. The length filter is applied again afterwards.

In [ ]:
mined_sentences = set()
with open(MINED, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        s = row["target"].strip()
        if s:
            mined_sentences.add(s)

print(f"{len(mined_sentences)} mined sentences to exclude")

In [ ]:
sents_removed, docs_affected, removed_short, kept = 0, 0, 0, 0

with open(CLEAN, encoding="utf-8") as infile, \
     open(NOLEAK, "w", encoding="utf-8") as outfile:
    for line in tqdm(infile):
        doc = json.loads(line)

        new_lines, removed = [], 0
        for ln in doc["text"].split("\n"):
            ln = ln.strip()
            if not ln:
                new_lines.append("")
                continue
            kept_sents = []
            for s in re.split(r'(?<=[.!?])\s+', ln):
                s = s.strip()
                if s in mined_sentences:
                    removed += 1
                else:
                    kept_sents.append(s)
            new_lines.append(" ".join(kept_sents))

        if removed:
            sents_removed += removed
            docs_affected += 1

        doc["text"] = "\n".join(new_lines)
        if len(doc["text"].split()) < MIN_WORDS:
            removed_short += 1
            continue

        outfile.write(json.dumps(doc, ensure_ascii=False) + "\n")
        kept += 1

print("Sentences removed:", sents_removed)
print("Documents affected:", docs_affected)
print("Removed (too short):", removed_short)
print("Kept:", kept)

## §4 Corpus size
in: `pmswiki_noleak.jsonl`

*Optional.* Whitespace-token counts. For the released file:

| documents | tokens | avg length | min | max |
|---:|---:|---:|---:|---:|
| 58,706 | 4,054,642 | 69.07 | 30 | 10,025 |

In [ ]:
lengths = []
with open(NOLEAK, encoding="utf-8") as f:
    for line in f:
        lengths.append(len(json.loads(line)["text"].split()))

print("Documents:", len(lengths))
print("Total tokens:", sum(lengths))
print("Average length:", round(sum(lengths) / len(lengths), 2))
print("Min length:", min(lengths))
print("Max length:", max(lengths))